In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

file_path = 'gistemp_global_mean.csv'

skip_rows = 0
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        if "-END HEADER-" in line:
            skip_rows = i + 1
            break

df = pd.read_csv(file_path, skiprows=skip_rows)
df.columns = [c.strip() for c in df.columns]

features_to_drop = ['YEAR', 'MO', 'DAY', 'DOY', 'MONTH']
df_filtered = df.drop(columns=[c for c in features_to_drop if c in df.columns])

df_numeric = df_filtered.select_dtypes(include=[np.number]).dropna()
df_clean = df_numeric.replace(-999, np.nan).dropna()

X_clean = df_clean.iloc[:, :-1]
y_clean = df_clean.iloc[:, -1]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X_train_c, y_train_c)

print(f'Retrained on {len(df_clean)} clean rows.')
print(f'New MSE: {mean_squared_error(y_test_c, final_model.predict(X_test_c))}')

Retrained on 145 clean rows.
New MSE: 0.0001252731034482777


In [2]:

df_clean = df_numeric.replace(-999, np.nan).dropna()

X_clean = df_clean.iloc[:, :-1]
y_clean = df_clean.iloc[:, -1]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)


final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X_train_c, y_train_c)

print(f'Retrained on {len(df_clean)} clean rows.')
print(f'New MSE: {mean_squared_error(y_test_c, final_model.predict(X_test_c))}')

Retrained on 145 clean rows.
New MSE: 0.0001252731034482777


In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    
    features = np.array(data['features']).reshape(1, -1)
    prediction = final_model.predict(features)
    # Added .item() to ensure JSON serialization of numpy floats
    return jsonify({'prediction': prediction[0].item(), 'target': 'GWETPROF'})

def run_app():
    # Set use_reloader=False if running inside Notebook environments
    app.run(port=5000, use_reloader=False, debug=False)

threading.Thread(target=run_app, daemon=True).start()
print("API is running locally on port 5000. Use POST /predict endpoint.")

API is running locally on port 5000. Use POST /predict endpoint.


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
